# Module 3: Sequential Chain

Apply **Pattern 1**: break the single-agent ceiling from Module 2 by running the same decision brief through a fixed 3-stage pipeline, Researcher → Analyst → Synthesizer, where each agent has one focused job and passes its output to the next.

**You** decide the order, in code: graph edges run the stages in the same sequence every time. In Module 4 you hand that decision to an orchestrator agent instead.

**When to use this pattern:**
- Steps have a natural, fixed order
- Each stage depends on the previous one
- You want simple, predictable, debuggable flow, with no routing decisions left to the model

> **Two models, on purpose.** The Researcher runs on Claude Sonnet 4.6; the Analyst and Synthesizer run on the smaller Amazon Nova Pro. Part 2 explains why.

**Key Strands primitive:** `GraphBuilder` (`strands.multiagent`).

**Prerequisites:** Modules 1 and 2. This module reuses Module 2's tools and the shared decision brief.

## Tools Used in This Module

| Tool | What it does | Key behavior |
|------|-------------|--------------|
| `get_company_data(company_name)` | Returns MarketNest financial/operational data | CLV, churn rate, revenue, top-spender metrics |
| `get_market_benchmarks(industry)` | Returns e-commerce benchmarks | Avg CLV, churn, subscription rates, CLV lift range |
| `get_competitor_data(competitor_name)` | Returns competitor premium tier data | Pricing, pilot approach, adoption %, CLV lift, payback period |

> **Note:** Only the **Researcher** agent has access to these tools. The Analyst and Synthesizer work with the text handed to them by the graph: they have no tool access.

In [ ]:
%pip install -r requirements.txt

In [ ]:
# ── Models ───────────────────────────────────────────────────────────────
# Researcher: no model= is passed, so it uses the Strands default on Amazon Bedrock,
#   Claude Sonnet 4.6 (global.anthropic.claude-sonnet-4-6). Tool use is the hard step.
# Analyst and Synthesizer: Amazon Nova Pro, deliberately a smaller model. They only
#   transform the text the graph hands them, so a cheaper model does the job.
# Nova Pro is called through its cross-region inference profile (us./eu./apac.):
# the bare "amazon.nova-pro-v1:0" is rejected on-demand in most regions.
# To try another model, replace NOVA_PRO, for example:
#   NOVA_PRO = BedrockModel(model_id="us.anthropic.claude-haiku-4-5-20251001-v1:0")

import os, sys, time
import boto3
sys.path.insert(0, os.path.join(os.getcwd(), "..", "02-single-agent"))
sys.path.insert(0, os.path.join(os.getcwd(), "..", "shared"))

from strands import Agent
from strands.models import BedrockModel
from strands.multiagent import GraphBuilder
from decision_brief_tools import get_company_data, get_market_benchmarks, get_competitor_data
from decision_brief import DECISION_BRIEF

region = os.environ.get("AWS_REGION") or boto3.Session().region_name
geo = {"us": "us", "eu": "eu", "ap": "apac"}.get(region.split("-")[0])
NOVA_PRO = BedrockModel(model_id=f"{geo}.amazon.nova-pro-v1:0") if geo else None  # None → Sonnet default

print("Tools loaded: get_company_data | get_market_benchmarks | get_competitor_data")
print("Analyst/Synthesizer model:", NOVA_PRO.config["model_id"] if NOVA_PRO else "Sonnet default (no Nova Pro profile in this region)")

---

## Part 1: Define the Prompts

Each agent gets a **narrow system prompt** focused on exactly one role. The docstring tells the model *when* to use a tool; the system prompt tells it *what job it has*.

The prompts also describe what each stage receives. A graph node gets the original task plus the output of the nodes that point to it, so the Synthesizer sees the brief and the Analyst's output, not the raw research. That is why the Analyst is asked to cite the numbers behind its verdicts.

In [ ]:
RESEARCHER_PROMPT = '''You are a market research specialist.
Given a decision brief, gather relevant company data, market benchmarks,
and competitor intelligence using your tools.
Return structured findings: data only, no recommendations.'''

ANALYST_PROMPT = '''You are a business strategy analyst.
You receive the decision brief and the Researcher's findings. Analyze each option (A, B, C).
For each option return:
- Strengths and weaknesses
- Implementation complexity: Low / Medium / High (with one-sentence justification)
- Top 2 risks with specific mitigations
- Verdict: Proceed / Proceed with caution / Do not proceed
Cite the research numbers behind each verdict: the next stage does not see the raw research.
Return structured analysis only, no executive memo yet.'''

SYNTHESIZER_PROMPT = '''You are an executive communications specialist.
You receive the decision brief and the Analyst's option analyses (which cite the research).
Write a leadership memo:

## Decision Memo: [Title]
**Recommendation**: [one sentence: which option and why]

### Options at a Glance
| | Option A | Option B | Option C |
|---|---|---|---|
| Complexity | | | |
| Risk level | | | |
| Verdict | | | |

### Top 3 Risks & Mitigations
### Success Metrics (3-5 KPIs with targets)
### Decision Required: owner · deadline · approvers needed

Under 400 words. Be direct.'''

---

## Part 2: Create the Three Agents

`callback_handler=None` makes every agent **silent** during graph execution. The graph runs all three; we display the final output after the chain completes.

A common production pattern with graphs is to run nodes on smaller models: the edges decide what happens next, so no model has to plan or route, and each node does one narrow, fixed job. Here the Analyst and Synthesizer get `model=NOVA_PRO` because they only transform text they are handed; the Researcher keeps the Sonnet default because choosing tools and their arguments is the hardest step.

In [ ]:
researcher = Agent(                      # Sonnet default: picks tools and their arguments
    tools=[get_company_data, get_market_benchmarks, get_competitor_data],
    system_prompt=RESEARCHER_PROMPT,
    callback_handler=None,
)

analyst = Agent(model=NOVA_PRO, system_prompt=ANALYST_PROMPT, callback_handler=None)

synthesizer = Agent(model=NOVA_PRO, system_prompt=SYNTHESIZER_PROMPT, callback_handler=None)

print("Agents created: researcher (Sonnet) | analyst (Nova Pro) | synthesizer (Nova Pro)")

---

## Part 3: Build the Graph and Run the Chain

`GraphBuilder` builds a **directed graph of agents**. Nodes are agents; edges define execution order. The graph passes each node's output to the nodes that depend on it.

For a short fixed sequence, plain function calls in your application also work (the Strands docs call this a workflow). The graph gives you more for the same few lines: an execution timeout, `execution_order`, per-node results and token usage, and room to add parallel or conditional edges later (see the optional modules in `samples/extra/`).

In [ ]:
builder = GraphBuilder()
builder.add_node(researcher,  "researcher")
builder.add_node(analyst,     "analyst")
builder.add_node(synthesizer, "synthesizer")
builder.add_edge("researcher", "analyst")      # researcher → analyst
builder.add_edge("analyst",    "synthesizer")  # analyst → synthesizer
builder.set_execution_timeout(300)             # stop a runaway graph (Strands warns when no limit is set)

print("Running: Researcher → Analyst → Synthesizer")
print("─" * 60)

t0 = time.time()
graph_result = builder.build()(DECISION_BRIEF)
elapsed = time.time() - t0

print(f"Chain complete in {elapsed:.1f}s  |  status: {graph_result.status.value}")
print("─" * 60)
print(graph_result.results["synthesizer"].result)

---

## Part 4: Inspect the Pipeline

In [ ]:
for node in graph_result.execution_order:
    print(f"=== {node.node_id.upper()} (first 300 chars) ===")
    print(str(node.result)[:300], "...")
    print()

print("Execution order:", " → ".join(n.node_id for n in graph_result.execution_order))
print()
print("What each agent saw:")
print("  Researcher:  the brief + its own tool results")
print("  Analyst:     the brief + the Researcher's output")
print("  Synthesizer: the brief + the Analyst's output")

In [ ]:
# Token usage per node: each agent has its own context and its own model, so you can see
# what each stage cost. Nova Pro tokens cost much less than Sonnet's:
# https://aws.amazon.com/bedrock/pricing/
models = {"researcher": researcher, "analyst": analyst, "synthesizer": synthesizer}
print(f"{'Stage':<14} {'Model':<34} {'Input':>8} {'Output':>8} {'Total':>8}")
print("-" * 76)
for node in graph_result.execution_order:
    u = node.result.accumulated_usage
    model_id = models[node.node_id].model.config["model_id"]
    print(f"{node.node_id:<14} {model_id:<34} {u['inputTokens']:>8,} {u['outputTokens']:>8,} {u['totalTokens']:>8,}")
u = graph_result.accumulated_usage
print("-" * 76)
print(f"{'graph total':<14} {'':<34} {u['inputTokens']:>8,} {u['outputTokens']:>8,} {u['totalTokens']:>8,}")

---

## Key Takeaways

| Concept | What you saw |
|---------|-------------|
| Sequential Chain | `GraphBuilder` defines the graph; edges fix the execution order |
| `add_node` / `add_edge` | Declarative graph definition, no manual Python orchestration |
| Output propagation | Each node gets the original task plus the output of the nodes before it |
| Focused context | Each agent sees only its inputs; per-node token usage shows what each stage cost |
| Right-sized models | Sonnet for the tool-using stage, the smaller Nova Pro for the text-only stages |
| Fixed order | Your edges pick the order, not the model; Module 4 hands that choice to an orchestrator |

**Strands primitive:** `GraphBuilder` (Graph)

---

## What's Next

In **Module 4: Agent-as-Tool**, an orchestrator agent decides which specialist to call and in what order, instead of graph edges.

Optional: in [Parallel Fork-Join](../extra/04-parallel-fork-join/) the analysts run at the same time instead of in sequence, with the same graph primitive and parallel edges.

---

## Want a real multi-turn conversation?

```bash
cd samples/03-sequential-chain
pip install -r requirements.txt
python chat.py
```